# Witness 2: Visible-but-useless spike (WP-A1)

**Gate served:** G0. **Preregistered in:** `model_card.md` Section 7.2 (pass rules frozen before execution).

**Mechanism tested (channel 2):** a strong supercritical factor orthogonal to the treated unit produces a clearly visible outlier eigenvalue while leaving counterfactual RMSE at the noise floor: the outlier carries zero usable signal for the treated unit.

**Design:** n = 120, T0 = 240, T_post = 100, sigma = 1. One donor-carried factor with s = 6 (predicted outlier at 1 + 6 + 0.5 + 0.5/6 = 7.583). Three arms share identical draws per replication (paired, base seed 50201, R = 400): NULL (all loadings zero), MISALIGNED (treated loading exactly 0), ALIGNED comparator (treated loading 3 s_d). Methods fit on pre-periods only: donor-mean, ridge-SC (intercept + CV penalty), simplex SCM (SLSQP), hard-threshold spectral SC (PC regression, k by largest eigenvalue-gap ratio).

**Pass rules:** Q1 coexistence (MISALIGNED mean top donor eigenvalue > 6.8 while NULL < 3.3); Q2 uselessness (per method: |mean paired RMSE diff| < 1 paired sd, and MISALIGNED mean <= 1.05 x NULL mean); Q3 sensitivity (ALIGNED spectral RMSE <= 1.05 sigma AND >= 10% below ALIGNED donor-mean, paired).

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import json, time
from pathlib import Path
import numpy as np
from scipy.optimize import minimize
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

N, T0, TPOST = 120, 240, 100
C_RATIO = N / T0
SIGMA = 1.0
EDGE = SIGMA * (1 + np.sqrt(C_RATIO)) ** 2
S_SPIKE = 6.0
SD2 = S_SPIKE * SIGMA ** 2 / (N - 1)
ALPHA_ALIGN = 3.0 * np.sqrt(SD2)
R_REPS = 400
BASE_SEED = 50201
K_MAX = 4
FIG_DIR = Path.cwd() / "figures"
FIG_DIR.mkdir(exist_ok=True)
print(f"edge={EDGE:.4f}, predicted outlier={(1 + S_SPIKE + C_RATIO + C_RATIO / S_SPIKE):.4f}, "
      f"s_d^2={SD2:.4f}, alpha_align={ALPHA_ALIGN:.4f}")

edge=2.9142, predicted outlier=7.5833, s_d^2=0.0504, alpha_align=0.6736


In [2]:
def pred_donor_mean(Dpre, ypre, Dpost):
    return Dpost.mean(axis=0)

def pred_ridge(Dpre, ypre, Dpost):
    nd, Tp = Dpre.shape
    X = np.hstack([np.ones((Tp, 1)), Dpre.T])
    XtX = X.T @ X
    Xty = X.T @ ypre
    lams = np.geomspace(1e-3, 1e3, 13) * np.trace(XtX) / (nd + 1)
    folds = np.array_split(np.arange(Tp), 4)
    best_lam, best_sse = None, np.inf
    for lam in lams:
        sse = 0.0
        for fidx in folds:
            mask = np.ones(Tp, dtype=bool)
            mask[fidx] = False
            Xf = X[mask]
            Af = Xf.T @ Xf
            Af[np.diag_indices_from(Af)] += lam
            Af[0, 0] -= lam
            bf = np.linalg.solve(Af, Xf.T @ ypre[mask])
            sse += float(((X[fidx] @ bf - ypre[fidx]) ** 2).sum())
        if sse < best_sse:
            best_sse, best_lam = sse, lam
    A = XtX.copy()
    A[np.diag_indices_from(A)] += best_lam
    A[0, 0] -= best_lam
    b = np.linalg.solve(A, Xty)
    Xp = np.hstack([np.ones((Dpost.shape[1], 1)), Dpost.T])
    return Xp @ b

def pred_simplex(Dpre, ypre, Dpost):
    nd = Dpre.shape[0]
    G = Dpre @ Dpre.T
    g = Dpre @ ypre
    yty = float(ypre @ ypre)
    def obj(w):
        return 0.5 * (w @ G @ w - 2.0 * w @ g + yty)
    def jac(w):
        return G @ w - g
    cons = {"type": "eq", "fun": lambda w: w.sum() - 1.0, "jac": lambda w: np.ones(nd)}
    res = minimize(obj, np.full(nd, 1.0 / nd), jac=jac, bounds=[(0.0, 1.0)] * nd,
                   constraints=cons, method="SLSQP",
                   options={"maxiter": 300, "ftol": 1e-10})
    return res.x @ Dpost

def pred_spectral(Dpre, ypre, Dpost):
    nd, Tp = Dpre.shape
    evals, evecs = np.linalg.eigh(Dpre @ Dpre.T / Tp)
    order = np.argsort(evals)[::-1]
    evals, evecs = evals[order], evecs[:, order]
    ratios = [evals[k] / evals[k + 1] for k in range(K_MAX)]
    k = int(np.argmax(ratios)) + 1
    V = evecs[:, :k]
    scores = Dpre.T @ V
    X = np.hstack([np.ones((Tp, 1)), scores])
    coef, *_ = np.linalg.lstsq(X, ypre, rcond=None)
    proj = Dpost.T @ V
    Xp = np.hstack([np.ones((Dpost.shape[1], 1)), proj])
    return Xp @ coef

METHODS = ["donor_mean", "ridge_sc", "simplex_scm", "spectral_ht"]
print("estimators defined")

estimators defined


In [3]:
t_start = time.time()
arms = ["null", "misaligned", "aligned"]
rmse = {a: {mm: np.empty(R_REPS) for mm in METHODS} for a in arms}
top_eig = {a: np.empty(R_REPS) for a in arms}

for j in range(R_REPS):
    rng = np.random.default_rng(BASE_SEED + j)
    f_pre = rng.normal(0.0, 1.0, size=T0)
    f_post = rng.normal(0.0, 1.0, size=TPOST)
    E_pre = rng.normal(0.0, SIGMA, size=(N, T0))
    E_post = rng.normal(0.0, SIGMA, size=(N, TPOST))
    a_don = rng.normal(0.0, np.sqrt(SD2), size=N)
    a_don[0] = 0.0
    loadings = {"null": np.zeros(N), "misaligned": a_don.copy(), "aligned": a_don.copy()}
    loadings["aligned"][0] = ALPHA_ALIGN
    for arm in arms:
        a = loadings[arm]
        Ypre = np.outer(a, f_pre) + E_pre
        y0_post = a[0] * f_post + E_post[0]
        Dpre, Dpost = Ypre[1:], np.outer(a[1:], f_post) + E_post[1:]
        ypre = Ypre[0]
        top_eig[arm][j] = np.linalg.eigvalsh(Dpre @ Dpre.T / T0)[-1]
        preds = {"donor_mean": pred_donor_mean(Dpre, ypre, Dpost),
                 "ridge_sc": pred_ridge(Dpre, ypre, Dpost),
                 "simplex_scm": pred_simplex(Dpre, ypre, Dpost)}
        preds["spectral_ht"] = pred_spectral(Dpre, ypre, Dpost)
        for mm in METHODS:
            rmse[arm][mm][j] = np.sqrt(float(((preds[mm] - y0_post) ** 2).mean()))
    if (j + 1) % 100 == 0:
        print(f"rep {j + 1}/{R_REPS} ({time.time() - t_start:.1f}s)")
print(f"all arms done in {time.time() - t_start:.1f}s")

rep 100/400 (148.6s)


rep 200/400 (282.7s)


rep 300/400 (411.4s)


rep 400/400 (544.7s)
all arms done in 544.7s


In [4]:
mis_top, null_top = top_eig["misaligned"].mean(), top_eig["null"].mean()
Q1 = bool(mis_top > 6.8 and null_top < 3.3)
print(f"Q1 coexistence : {'PASS' if Q1 else 'FAIL'} "
      f"(NULL mean top = {null_top:.3f}, MISALIGNED mean top = {mis_top:.3f}; "
      f"edge = {EDGE:.3f}, predicted outlier = {1 + S_SPIKE + C_RATIO + C_RATIO / S_SPIKE:.3f})")

q2_lines, Q2_parts = [], {}
for mm in METHODS:
    diff = rmse["misaligned"][mm] - rmse["null"][mm]
    ok = bool(abs(diff.mean()) < diff.std(ddof=1)
              and rmse["misaligned"][mm].mean() <= 1.05 * rmse["null"][mm].mean())
    Q2_parts[mm] = ok
    q2_lines.append(f"  {mm:12s}: mean diff = {diff.mean():+.4f}, paired sd = {diff.std(ddof=1):.4f}"
                    f", rel MIS/NULL = {rmse['misaligned'][mm].mean() / rmse['null'][mm].mean():.4f}"
                    f" -> {'PASS' if ok else 'FAIL'}")
Q2 = bool(all(Q2_parts.values()))
print("Q2 uselessness :", "PASS" if Q2 else "FAIL")
print("\n".join(q2_lines))

aln_spec = rmse["aligned"]["spectral_ht"].mean()
aln_dm = rmse["aligned"]["donor_mean"].mean()
paired_gain = rmse["aligned"]["donor_mean"] - rmse["aligned"]["spectral_ht"]
gain_t = paired_gain.mean() / (paired_gain.std(ddof=1) / np.sqrt(R_REPS))
Q3 = bool(aln_spec <= 1.05 * SIGMA and aln_spec <= 0.90 * aln_dm and gain_t > 4.0)
print(f"Q3 sensitivity : {'PASS' if Q3 else 'FAIL'} "
      f"(ALIGNED spectral RMSE = {aln_spec:.4f} vs donor-mean {aln_dm:.4f}, sigma = {SIGMA}, gain t = {gain_t:.1f})")

W2_PASS = bool(Q1 and Q2 and Q3)
print("WITNESS 2 OVERALL :", "PASS" if W2_PASS else "FAIL")

Q1 coexistence : PASS (NULL mean top = 2.820, MISALIGNED mean top = 7.607; edge = 2.914, predicted outlier = 7.583)
Q2 uselessness : PASS
  donor_mean  : mean diff = +0.0002, paired sd = 0.0022, rel MIS/NULL = 1.0002 -> PASS
  ridge_sc    : mean diff = -0.0001, paired sd = 0.0039, rel MIS/NULL = 0.9999 -> PASS
  simplex_scm : mean diff = +0.0007, paired sd = 0.0061, rel MIS/NULL = 1.0007 -> PASS
  spectral_ht : mean diff = +0.0005, paired sd = 0.0082, rel MIS/NULL = 1.0005 -> PASS
Q3 sensitivity : PASS (ALIGNED spectral RMSE = 1.0395 vs donor-mean 1.2102, sigma = 1.0, gain t = 60.3)
WITNESS 2 OVERALL : PASS


In [5]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
bins = np.linspace(2.4, 8.4, 40)
ax.hist(top_eig["null"], bins=bins, alpha=0.6, color="#7f7f7f", label="NULL arm")
ax.hist(top_eig["misaligned"], bins=bins, alpha=0.6, color="#1f77b4", label="MISALIGNED arm")
ax.axvline(EDGE, color="k", ls=":", lw=1.2, label=f"MP edge {EDGE:.2f}")
out_pred = 1 + S_SPIKE + C_RATIO + C_RATIO / S_SPIKE
ax.axvline(out_pred, color="crimson", ls="--", lw=1.2, label=f"BGN outlier {out_pred:.2f}")
ax.set_xlabel(r"top eigenvalue of $(1/T_0) D D^\top$, donors only")
ax.set_ylabel("replications")
ax.set_title("Witness 2: a large outlier coexists with zero usable signal for unit 1")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig_w2_eigenvalue_outlier.png", dpi=150)
plt.close(fig)

fig, ax = plt.subplots(figsize=(9.5, 4.2))
positions, data, colors, labels = [], [], [], []
pal = {"null": "#7f7f7f", "misaligned": "#1f77b4", "aligned": "#2ca02c"}
pos = 0
for mm in METHODS:
    for arm in arms:
        pos += 1
        positions.append(pos)
        data.append(rmse[arm][mm])
        colors.append(pal[arm])
        labels.append(mm)
bp = ax.boxplot(data, positions=positions, widths=0.65, showfliers=False, patch_artist=True)
for patch, cc in zip(bp["boxes"], colors):
    patch.set_facecolor(cc)
    patch.set_alpha(0.7)
ax.axhline(SIGMA, color="crimson", ls="--", lw=1.1, label=r"oracle floor $\sigma$")
ax.set_xticks(positions)
ax.set_xticklabels(labels, rotation=0, fontsize=8)
ax.set_ylabel("post-period RMSE / sigma")
ax.set_title("Witness 2: RMSE by method and arm (grey NULL, blue MISALIGNED, green ALIGNED)")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig_w2_rmse_distributions.png", dpi=150)
plt.close(fig)
print("figures saved")

figures saved


In [6]:
summary = dict(witness="w2_misalignment",
               config=dict(n=N, T0=T0, T_post=TPOST, c=C_RATIO, sigma=SIGMA, s_spike=S_SPIKE,
                           alpha_align=float(ALPHA_ALIGN), R=R_REPS, base_seed=BASE_SEED),
               results=dict(mean_top_eig={a: float(top_eig[a].mean()) for a in arms},
                            mean_rmse={a: {mm: float(rmse[a][mm].mean()) for mm in METHODS}
                                       for a in arms}),
               verdicts=dict(Q1=Q1, Q2=Q2_parts, Q3=Q3),
               overall_pass=W2_PASS)
with open(FIG_DIR / "witness_w2_summary.json", "w") as fh:
    json.dump(summary, fh, indent=2)
print(json.dumps(summary["results"], indent=2))

{
  "mean_top_eig": {
    "null": 2.820126321166871,
    "misaligned": 7.6068938486621365,
    "aligned": 7.6068938486621365
  },
  "mean_rmse": {
    "null": {
      "donor_mean": 1.0021570912032962,
      "ridge_sc": 1.0030560088224825,
      "simplex_scm": 1.0294500538266886,
      "spectral_ht": 1.002278537587321
    },
    "misaligned": {
      "donor_mean": 1.0023822782475043,
      "ridge_sc": 1.0029969131563545,
      "simplex_scm": 1.0301202391372053,
      "spectral_ht": 1.0027783427144459
    },
    "aligned": {
      "donor_mean": 1.2102362633366075,
      "ridge_sc": 1.0758769826431815,
      "simplex_scm": 1.1060037682792148,
      "spectral_ht": 1.0394527964427842
    }
  }
}


## Interpretation template

If Q1-Q3 pass: an economically large factor (outlier far above the MP edge) contributes nothing to the treated unit's counterfactual when the treated loading is exactly zero (channel 2 confirmed), while the aligned comparator shows the battery detects genuinely useful spikes. Failure readings are documented in `model_card.md` Section 7.2.